"Rooted trees (...) 'encode' the nature of hierarchy". They abide by a trifecta:
* G is connected
* G does not contain a cycle
* G has n-1 edges
**where any two statements imply the third**
(pg. 78)

In [15]:
from dataclasses import dataclass


e = [(2,1),(2,3),(4,2),(4,6),(6,5),(6,7)]
n = [1,2,3,4,5,6,7]

@dataclass
class Graph:
    edges: list[tuple[int, int]]
    nodes: list[int]
    V: int = 0
    E: int = 0

    def __post_init__(self):
        V = len(self.nodes)
        E = len(self.edges)

my_tree = Graph(edges=e, nodes=n)

## Graph connectivity and graph traversal
"Suppose we are given a graph G = (V, E) and two particular nodes s and t. We'd like (...) an efficient algorithm that answers (...): is there a path from s to t in G?" (pg. 78)

"(...) the s-t Connectivity Problem could also be called the Maze-Solving Problem." (pg. 78)

## Breadth-first Search (BFS)
"Perhaps the simplest algorithm for determining s-t connectivity (...) we explore outward from s in all possible directions, adding nodes one “layer” at a time." (pg. 79)

"... there is a natural physical interpretation to the
algorithm. Essentially, we start at s and “flood” the graph with an **expanding
wave** that grows to visit all nodes that it can reach." (pg. 79)



In [16]:
import random, copy
from collections import deque


# starting at a given node, go through all edges to find new nodes.
# repeat for every found node. O(nm). Expensive.
def bfs(G: Graph, root_node_index: int) -> list[int]: 
    discovered = set()
    q = deque()
    root_node = G.nodes[root_node_index]

    nodes_found = list[int]()
    # edges = copy.deepcopy(G.edges)
    q.append(root_node)
    discovered.add(q[0])
    while len(q) > 0:
        current_node = q[0]
        for edge in G.edges:
            if edge[0] == current_node and edge[1] not in discovered:
                discovered.add(edge[1])
                q.append(edge[1])
            elif edge[1] == current_node and edge[0] not in discovered:
                discovered.add(edge[0])
                q.append(edge[0])
        nodes_found.append(current_node)
        q.popleft()
    return nodes_found

assert(bfs(my_tree,5)==[6,4,5,7,2,1,3])
assert(bfs(my_tree,6)==[7,6,4,5,2,1,3])


my_tree_bfs = Graph(edges=[(1,2),(1,3), (2,4), (2,5), (3,6), (3,7), (4,8), (4,9), (5,10), 
                       (5,11),(6,12),(6,13),(7,14), (7,15)],
                nodes=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15])

rand_root_node_index = random.randrange(0,len(my_tree_bfs.nodes)+1)

print("Root node: ", my_tree_bfs.nodes[rand_root_node_index])
print("  BFS: ", bfs(my_tree_bfs, rand_root_node_index))
print("Root node: 1")
print("  BFS: ", bfs(my_tree_bfs, 0))



Root node:  9
  BFS:  [9, 4, 2, 8, 1, 5, 3, 10, 11, 6, 7, 12, 13, 14, 15]
Root node: 1
  BFS:  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


"Layer L<sub>1</sub> consists of all nodes that are neighbors of *s*" (pg. 80) where *s* is some starting node (in essence, the "new" root). 

"Assuming that we have defined layers L<sub>1</sub>, ..., L<sub>j</sub>, then layer L<sub>j+1</sub> consists of all nodes that do not belong to an earlier layer and that have an edge to a node in layer L<sub>j</sub>." (pg. 80)

L<sub>j</sub> is the set of all nodes at distance *j* from *s*. A node that does not appear in BFS's output fis a node with no path to it. It follows then that BFS does not only show what nodes are reachable starting at s, but also the shortest paths to them. (pg. 80).

"**(3.3)** For each J >= 1, layer L<sub>j</sub> produced by BFS consists of all nodes at distance exactly *j* from *s*. There is a path from *s* to *t* if and only if *t* appears in some layer." (pg. 80)

"The set of nodes discovered by the BFS algorithm is precisely those reachable from the starting node *s*. We will refere to this set *R* as the *connected component* of *G* containing *s*" (pg. 82)

In [17]:
# We can use some helper classes to build our adjecency list...
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None
class LinkedList:
    def __init__(self):
        self.head = None  # The entry point of the list
        self.tail = None
    # Add a node at the end of the list
    def append(self, data: int):
        new_node = Node(data)
        if not self.head:
            self.head = new_node
            self.tail = new_node
            return
        current = self.head
        while current.next:  # Traverse to the last node
            current = current.next
        current.next = new_node
        self.tail = new_node

# both creating the adjancency list and walking it for a BFS tree take O(m+n).
# the exact details in the book aside, the key is that for every node visited _u_, 
# its incident n_u edges are checked. The sum of incident edges is 2m, so the total
# time checking all incident edges is O(m). In a connected graph, n nodes are inspected
# in our outer loop. So the time complexity is O(m+n)
def bfs2(G: Graph, starting_node: int) -> list[int]:
    # first we initialize our adj_list "empty"
    adj_list = dict[int, LinkedList]((node, LinkedList()) for node in G.nodes)

    # Now we fill the adjencency list
    for edge in G.edges:
        if edge[0] in adj_list:
            new_node = Node(edge[1])
            list_head = adj_list[edge[0]].head
            prev_head = None
            while list_head is not None and list_head.data is not edge[1]:
                prev_head = list_head
                list_head = list_head.next
            if list_head is None:
                if adj_list[edge[0]].tail:
                    adj_list[edge[0]].tail.next = new_node
                    adj_list[edge[0]].tail = adj_list[edge[0]].tail.next
                else:
                    adj_list[edge[0]].append(edge[1])
        if edge[1] in adj_list:
            new_node = Node(edge[0])
            list_head = adj_list[edge[1]].head
            prev_head = None
            while list_head is not None and list_head.data is not edge[0]:
                prev_head = list_head
                list_head = list_head.next
            if list_head is None:
                if adj_list[edge[1]].tail:
                    adj_list[edge[1]].tail.next = new_node
                    adj_list[edge[1]].tail = adj_list[edge[1]].tail.next
                else:
                    adj_list[edge[1]].append(edge[0])
    discovered = list()
    q = deque[int]()
    v = set[int]()
    q.append(starting_node)
    v.add(q[0])
    # print(adj_list[5].head)
    while len(q) > 0:
        c_node = q[0]
        list_head = adj_list[c_node].head
        while list_head is not None:
            if (list_head.data not in v):
                v.add(list_head.data)
                q.append(list_head.data)
            list_head = list_head.next
        discovered.append(c_node)
        q.popleft()
    return discovered

assert(set(bfs2(my_tree,5))==set([6,4,5,7,2,1,3]))
assert(set(bfs2(my_tree_bfs, 6))==set([7, 3, 14, 15, 1, 6, 2, 12, 13, 4, 5, 8, 9, 10, 11]))



In [18]:
import time
import pandas as pd

my_tree_bfs = Graph(edges=[(1,2),(1,3), (2,4), (2,5), (3,6), (3,7), (4,8), (4,9), (5,10), 
                       (5,11),(6,12),(6,13),(7,14), (7,15),(8,16),(8,17),(9,18),(9,19),
                       (10,20),(10,21),(11,22),(11,23),(12,24),(12,25),(13,26),(13,27),
                       (14,28),(14,29),(15,30),(15,31),
                       (1,15),(2,21),(3,29),(3,27),(4,11),(4,29),(4,14),(5,6),(5,31),(5,30),
                       (5,19),(5,1),(6,10),(6,31),(6,30),(6,27),(7,1),(7,11)],
                nodes=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,
                       26,27,28,29,30,31])

# For these m > n, so O(m + n) = O(m), and O(m*n)=O(m^2)

root_indices = list[list[int]]()
for i in range(37): 
    root_indices.append([random.randrange(1, len(my_tree_bfs.nodes)) for _ in range(37)])

first_run_total = 0
print("Running first bfs(): ")
for roots in root_indices:
    start = time.perf_counter()
    for j in roots:
        bfs(my_tree_bfs,j)
    end = time.perf_counter()
    run_time = end - start
    first_run_total += run_time
first_run_total_str = f"{first_run_total:.6f}"
print(f"    Total: {first_run_total_str} seconds")

second_run_total = 0
print("Running second bfs(): ")
for roots in root_indices:
    start = time.perf_counter()
    for j in roots:
        bfs2(my_tree_bfs,j)
    end = time.perf_counter()
    run_time = end - start
    second_run_total += run_time
second_run_total_str = f"{second_run_total:.6f}"
print(f"    Total: {second_run_total_str} seconds")
# print(pd.Series(el for sublist in root_indices for el in sublist).value_counts())

# however, it looks like this 'm' is not big enough to show that bfs2 is faster than bfs


Running first bfs(): 
    Total: 0.121052 seconds
Running second bfs(): 
    Total: 0.077676 seconds


# Depth-First Search
There is a path from the starting node passed to DFS to all the nodes returned by the function call:
    "(3.6) For a given recursive call DFS(u), all nodes that are marked “Explored” between the invocation and end of this recursive call are descendants of *u* in *T*." (pg. 85)

If (*x*,*y*) is in *G* but not in *T*, and *x* and *y* are in *T*, then *x* and *y* are an ancestor and descendant of each other:
    "(3.7) Let T be a depth-first search tree, let x and y be nodes in T, and let(x, y) be an edge of G that is not an edge of T. Then one of x or y is an ancestor of the other." (pg. 85)
and most importantly in my opinion, since (3.7) holds for any *x* and *y* nodes in *T*, *T* is connected. 

In [19]:

def dfs(G: Graph, starting_node: int) -> tuple[list[int], list[tuple[int,int]]]:
    # first we initialize our adj_list "empty"
    adj_list = dict[int, LinkedList]((node, LinkedList()) for node in G.nodes)

    # Now we fill the adjencency list
    for edge in G.edges:
        if edge[0] in adj_list:
            new_node = Node(edge[1])
            list_head = adj_list[edge[0]].head
            prev_head = None
            while list_head is not None and list_head.data is not edge[1]:
                prev_head = list_head
                list_head = list_head.next
            if list_head is None:
                if adj_list[edge[0]].tail:
                    adj_list[edge[0]].tail.next = new_node
                    adj_list[edge[0]].tail = adj_list[edge[0]].tail.next
                else:
                    adj_list[edge[0]].append(edge[1])
        if edge[1] in adj_list:
            new_node = Node(edge[0])
            list_head = adj_list[edge[1]].head
            prev_head = None
            while list_head is not None and list_head.data is not edge[0]:
                prev_head = list_head
                list_head = list_head.next
            if list_head is None:
                if adj_list[edge[1]].tail:
                    adj_list[edge[1]].tail.next = new_node
                    adj_list[edge[1]].tail = adj_list[edge[1]].tail.next
                else:
                    adj_list[edge[1]].append(edge[0])
    found = list()
    tree = list[tuple[int,int]]()
    q = deque[int]()
    explored = set[int]()
    q.append(starting_node)
    parent = dict[int,int]()
    # print(adj_list[5].head)
    parent[q[0]] = None
    while len(q) > 0:
        c_node = q[-1]
        if c_node not in explored:
            explored.add(c_node) # add to visited _when_ processing starts
            if(parent[c_node]):
                tree.append((c_node,parent[c_node]))
                found.insert(0, c_node)
            list_head = adj_list[c_node].head
            while list_head is not None:
                q.append(list_head.data)
                parent[list_head.data] = c_node
                list_head = list_head.next
        else:
            q.pop()
    found.append(starting_node)
    #found.append(starting_node)
    return found, tree

# assert(set(dfs(my_tree,5))==set([6,4,5,7,2,1,3]))
print(dfs(my_tree,5)[0])
print(dfs(my_tree,5)[1])

[1, 3, 2, 4, 7, 6, 5]
[(6, 5), (7, 6), (4, 6), (2, 4), (3, 2), (1, 2)]


## Testing Bipartiteness
    "(3.14) If a graph G is bipartite, then it cannot contain an odd cycle." (pg. 95)
Odd cycles cannot be colored in an alternating fashion with two colors. 

BFS can be used to detect whether some Graph G is bipartate. 
First we asume G is connected, otherwise we can find G's connected components and analyze each individually. (pg. 95)
Then we can call BFS on some *s* of G and color *s* the first color, and every node in each layer i+1 (since s is L<sub>i = 0 </sub>) a single color different than the previous layer's. 

The following claim characterizes the only two possible results:<br>
"**(3.15)** Let G be a connected graph, and let L1, L2, . . . be the layers produced by BFS starting at node s. Then exactly one of the following two things must hold.
* (i) There is no edge of G joining two nodes of the same layer. In this case G is a bipartite graph in which the nodes in even-numbered layers can be
colored red, and the nodes in odd-numbered layers can be colored blue.
* (ii) There is an edge of G joining two nodes of the same layer. In this case, G contains an odd-length cycle, and so it cannot be bipartite." (pg. 96)

If (i), every edge has nodes of different colors, and G is bipartate.
if (ii), the two nodes in the same layer are connected through an odd number of ancestors, so there is an odd cycle.

In [23]:
def isBipartate(G: Graph, root_node_index: int = 0) -> bool:
    color = dict[int,int]()
    discovered = set[int]()
    q = deque()
    root_node = G.nodes[root_node_index]

    nodes_found = list[int]()
    # edges = copy.deepcopy(G.edges)
    q.append(root_node)
    discovered.add(q[0])
    color[q[0]] = 0 # 0 is "first" color, "red", etc. 
    while len(q) > 0:
        current_node = q[0]
        current_color = color[current_node]
        for edge in G.edges:
            if edge[0] == current_node:
                if edge[1] not in discovered:
                    discovered.add(edge[1])
                    q.append(edge[1])
                    color[edge[1]] = 0 if current_color == 1 else 1
                else:
                    if color[edge[1]] == color[edge[0]]:
                        return False
            elif edge[1] == current_node:
                if edge[0] not in discovered:
                    discovered.add(edge[0])
                    q.append(edge[0])
                    color[edge[0]] = 0 if current_color == 1 else 1
                else:
                    if color[edge[0]] == color[edge[1]]:
                        return False
        nodes_found.append(current_node)
        q.popleft()
    # if whole tree is built without odd cycles
    return True

my_BP_tree = Graph(edges=[(1,2),(1,3), (2,4),(2,5), (3,5),(3,6)],
                nodes=[1,2,3,4])
assert(isBipartate(my_BP_tree)==True)
assert(isBipartate(my_BP_tree, random.randrange(0, len(my_BP_tree.nodes)))==True)
my_BP_tree.edges=[(1,2),(1,3),(2,4),(2,5),(3,5),(3,6),(4,7),(4,8),(5,8),(5,9),(6,9),(6,10)]
my_BP_tree.nodes=[1,2,3,4,5,6,7,8,9,10]
assert(isBipartate(my_BP_tree)==True)
assert(isBipartate(my_BP_tree, random.randrange(0, len(my_BP_tree.nodes)))==True)
my_BP_tree.edges.append((2,3))
assert(isBipartate(my_BP_tree)==False)
assert(isBipartate(my_BP_tree, random.randrange(0, len(my_BP_tree.nodes)))==False)
my_BP_tree.edges.pop()
my_BP_tree.edges.append((4,5))
assert(isBipartate(my_BP_tree)==False)
assert(isBipartate(my_BP_tree, random.randrange(0, len(my_BP_tree.nodes)))==False)
my_BP_tree.edges.pop()
my_BP_tree.edges.append((7,8))
assert(isBipartate(my_BP_tree)==False)
assert(isBipartate(my_BP_tree, random.randrange(0, len(my_BP_tree.nodes)))==False)
my_BP_tree.edges.pop()
my_BP_tree.edges.remove((2,5))
my_BP_tree.edges.remove((3,5))
assert(isBipartate(my_BP_tree)==True)
assert(isBipartate(my_BP_tree, random.randrange(0, len(my_BP_tree.nodes)))==True)